# From Cold Pool to Hot Queue

Turn a 1,754-image unlabeled pool into a fine-tuned detector, end to end. Run cells top to bottom. Steps with no code (Annotate, Crop Query panel) show up as a note instead of a code cell.

In [ ]:
!pip install -q fiftyone requests transformers torch huggingface_hub umap-learn open-clip-torch

In [ ]:
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.zoo as foz
import fiftyone.operators as foo
from fiftyone import ViewField as F

## Load the pool

Fast path: pull the exact 1,754-image, media-only pool from Hugging Face Hub. (To rebuild it from source instead, run the numbered `01_`-`04_` scripts in this folder.)

In [ ]:
from fiftyone.utils.huggingface import load_from_hub

dataset = load_from_hub("harpreetsahota/InsPLAD-workshop-pool")
session = fo.launch_app(dataset)

## Step 1: Verify it's unlabeled, define the working scope

`interactive` excludes the 100 `eval_holdout` samples — every search, scoring, and annotation call below uses it, not `dataset`.

In [ ]:
print(dataset.get_field_schema())  # no label fields yet, only filepath/tags/metadata

interactive = dataset.match_tags("eval_holdout", bool=False)  # 1,654 images

## Step 2: Embed the pool once with CLIP

Every brain method below reuses this same `clip_embedding` field.

In [ ]:
model = foz.load_zoo_model("clip-vit-base32-torch")
dataset.compute_embeddings(model, embeddings_field="clip_embedding")

## Step 3: Find and tag near-duplicate images

Tags, doesn't delete — 73/1,754 flagged on this pool. Deleting would quietly shrink whichever class happens to have the most duplicate-flight coincidences.

In [ ]:
sim_index = fob.compute_similarity(dataset, embeddings="clip_embedding", brain_key="clip_dup_sim")
sim_index.find_duplicates(thresh=0.10)

dataset.select(sim_index.duplicate_ids).tag_samples("near_duplicate")
review_view = sim_index.duplicates_view()  # review pairs side by side before trusting the tag

## Step 4: Visualize with UMAP, score uniqueness + representativeness

`uniqueness`/`representativeness` are scored against `interactive` (not `dataset`) — they feed Step 8's prioritization directly, and the eval holdout has no business entering that ranking. Open the Embeddings panel in the App and lasso a cluster to explore.

In [ ]:
fob.compute_visualization(
    dataset,
    embeddings="clip_embedding",
    method="umap",
    brain_key="pool_clip_umap",
)

fob.compute_uniqueness(interactive, embeddings="clip_embedding")
fob.compute_representativeness(
    interactive,
    embeddings="clip_embedding",
    method="cluster-center-downweight",
)

## Step 5: Test text-based similarity search

Real lift over random, not good enough alone (25/100 for `"tower id plate"` vs. 12.9% random baseline on this pool). Try it on your own class name before assuming it'll work.

In [ ]:
fob.compute_similarity(
    interactive, model="clip-vit-base32-torch", embeddings="clip_embedding", brain_key="clip_text_sim"
)

text_search_view = interactive.sort_by_similarity(
    "tower id plate", k=100, brain_key="clip_text_sim"
)

## Step 6: Search with a few labeled examples instead of a word

Browse the grid, find 3 clear examples of your class, copy their sample IDs from each sample's modal.

In [ ]:
seed_ids = ["<id_1>", "<id_2>", "<id_3>"]  # fill in 3 real sample IDs

view = interactive.sort_by_similarity(seed_ids, k=100, brain_key="clip_text_sim")

# Tag confirmed hits so Step 8 knows not to waste budget re-surfacing them:
# view.select(<confirmed_ids>).tag_samples("potential_match")

**No-code alternative:** the [Crop Query](https://github.com/harpreetsahota204/crop_query) panel does the same seeded search at the patch level, with a localizing heatmap, directly in the App:
```
fiftyone plugins download https://github.com/harpreetsahota204/crop_query
```

## Step 7: Try a second embedding backbone: C-RADIO

Different models see different structure in the same pixels — `yoke` goes from one dense CLIP blob to visible sub-clusters in C-RADIO.

In [ ]:
foz.register_zoo_model_source("https://github.com/harpreetsahota204/NVLabs_CRADIOV3")

radio_model = foz.load_zoo_model("NVLabs_CRADIOV3")
dataset.compute_embeddings(radio_model, embeddings_field="radio_embedding")

fob.compute_visualization(
    dataset,
    embeddings="radio_embedding",
    method="umap",
    brain_key="pool_radio_umap",
)

fob.compute_uniqueness(interactive, embeddings="radio_embedding", uniqueness_field="radio_uniqueness")
fob.compute_representativeness(
    interactive,
    embeddings="radio_embedding",
    representativeness_field="radio_rep",
    method="cluster-center-downweight",
)

## Step 8: Prioritize what's left with all four signals

Excluding `potential_match` *before* ranking (not after) matters: blending naively across the whole pool re-surfaces images Step 6 already found.

In [ ]:
import numpy as np

def minmax(a):
    a = np.array(a)
    return (a - a.min()) / (a.max() - a.min())

not_yet_found = interactive.match_tags("potential_match", bool=False)

triage_score = (
    0.25 * minmax(not_yet_found.values("uniqueness"))
    + 0.25 * minmax(not_yet_found.values("representativeness"))
    + 0.25 * minmax(not_yet_found.values("radio_uniqueness"))
    + 0.25 * minmax(not_yet_found.values("radio_rep"))
)
not_yet_found.set_values("triage_score", triage_score.tolist())

hot_queue = not_yet_found.sort_by("triage_score", reverse=True)
dataset.save_view(
    "hot_queue_triage",
    hot_queue,
    description="Hot queue sorted by blended triage_score.",
)

## Step 9: Annotate in the FiftyOne App

No code, on purpose. One-time setup: add an Annotation Schema (`human_annotated` as a `Detections` field, the 4 target classes). Then work the `hot_queue_triage` view — open each sample, switch to the "Annotate" tab, draw boxes. Auto-saves to `human_annotated`, never `ground_truth`.

## Step 10: Fine-tune a detector with RF-DETR

Requires the [hf_fine_tuner_plugin](https://github.com/harpreetsahota204/hf_fine_tuner_plugin): `fiftyone plugins download https://github.com/harpreetsahota204/hf_fine_tuner_plugin`.

In [ ]:
det_trainer = foo.get_operator(
    "@harpreetsahota/hf_fine_tuner_plugin/finetune_detection"
)

# Defaults (3 epochs, lr=1e-5) undertrain this: RF-DETR's classification
# head is reinitialized from scratch for these 4 classes (pretrained on
# 91 COCO classes), so it needs more steps and a higher LR to converge.
# load_best_model_at_end=True keeps the best checkpoint regardless.
det_trainer(
    interactive.match(F("human_annotated.detections").length() > 0),
    label_field="human_annotated",
    model_name="Roboflow/rf-detr-base",
    bbox_format="coco",
    num_epochs=75,
    learning_rate=5e-5,
    batch_size=16,
    delegate=True,
)

## Step 11: Spot-check the model on what it hasn't seen yet

No `ground_truth` out here to score against — this is a vibe check, not a metric. Scroll predictions class by class in the App; where a class looks weak, annotate a few more examples and re-run Step 10.

In [ ]:
from transformers import AutoModelForObjectDetection

not_yet_annotated = interactive.exists("human_annotated", bool=False)
model = AutoModelForObjectDetection.from_pretrained("finetuned_detection_model")  # wherever finetune_detection saved its output_dir

# confidence_thresh matters: RF-DETR always emits 300 raw query predictions
# per image, most of them near-zero-confidence noise.
not_yet_annotated.apply_model(model, label_field="predictions", confidence_thresh=0.5)

session.view = not_yet_annotated

## Step 12: Evaluate against the held-out labels

Once satisfied with Step 11's coverage. Run `08_reveal_eval_holdout_ground_truth.py` first (one time) to populate `ground_truth` on the 100 `eval_holdout` samples and build the `eval_holdout_annotated` saved view.

In [ ]:
eval_view = dataset.load_saved_view("eval_holdout_annotated")
eval_view.apply_model(model, label_field="predictions", confidence_thresh=0.5)

results = eval_view.evaluate_detections("predictions", gt_field="ground_truth")
results.print_report()